# 05 - Export app data

Writes `app_data/app_data.json`, the single file the React app consumes.

Each destination carries:
- Identity (city, country, iso2, lat/lon, categories, tags, blurb)
- Routing info (anchor airport, transfer warning, no_ryanair_route flag)
- `routes` per origin: daily outbound/return Ryanair fare calendars
- `costs`: the on-the-ground lifestyle price basket for that location
- `accommodation`: the Airbnb nightly anchor for that location

Pricing logic (flights + accommodation + lifestyle) and the Skyscanner deeplink
live in the app's own `continent-app/src/runtime_pricing.js`. The accommodation
fee/seasonality/discount parameters travel in `meta.accommodation_model`. See
`SCHEMA.md` for the full contract.

## 1. Load combined dataset

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

CACHE_DIR = Path("cache")
APP_DIR   = Path("app_data")
APP_DIR.mkdir(exist_ok=True)

cfg      = json.loads((CACHE_DIR / "config.json").read_text(encoding="utf-8"))
combined = json.loads((CACHE_DIR / "combined_dataset.json").read_text(encoding="utf-8"))
master   = json.loads((CACHE_DIR / "destinations_master.json").read_text(encoding="utf-8"))
costs    = json.loads((CACHE_DIR / "costs.json").read_text(encoding="utf-8"))
accom    = json.loads((CACHE_DIR / "accommodation.json").read_text(encoding="utf-8"))

destinations = combined["destinations"]
print(f"Loaded {len(destinations)} destinations")

Loaded 450 destinations


## 2. Write app_data.json

In [2]:
APP_JSON = APP_DIR / "app_data.json"

payload = {
    "meta": {
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "schema_version": cfg["schema_version"],
        "currency":       cfg["currency"],
        "origins":        cfg["origins"],
        "home_city":      cfg["home_city"],
        "home":           {"lat": cfg["home_lat"], "lon": cfg["home_lon"]},
        "start_date":     cfg["start_date"],
        "end_date":       cfg["end_date"],
        "categories":     master["meta"]["categories"],
        "defaults": {
            "group_size":         cfg["group_size_default"],
            "trip_length_days":   cfg["trip_length_days_default"],
            "baggage":            cfg["baggage_default"],
            "lifestyle":          cfg["lifestyle_defaults"],
        },
        "baggage_options":      cfg["baggage_options"],
        "cost_basket":          costs["meta"]["basket"],        # item -> human label
        "cost_validation":      costs["meta"]["validation"],    # accuracy of the cost model
        "accommodation_model":  accom["meta"]["model"],         # fees/seasonality/discount params
        "accommodation_validation": accom["meta"]["validation"],# accuracy of the accom model
        "n_destinations":  len(destinations),
        "is_mock":         False,
    },
    "destinations": destinations,
}

APP_JSON.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
size_mb = APP_JSON.stat().st_size / (1024 * 1024)
print(f"Wrote {APP_JSON}  ({size_mb:.2f} MB)")

Wrote app_data\app_data.json  (0.59 MB)


## 3. Sanity check

In [3]:
sample = next(iter(destinations.values()))
print(f"Sample destination: {sample['city']}, {sample['country']}")
print(f"  Keys: {sorted(sample.keys())}")

with_fares = sum(1 for d in destinations.values() if d["routes"])
with_costs = sum(1 for d in destinations.values() if d["costs"])
with_accom = sum(1 for d in destinations.values() if d.get("accommodation"))
no_route   = sum(1 for d in destinations.values() if d["no_ryanair_route"])
print(f"\nDestinations with fare routes: {with_fares}")
print(f"Destinations with cost basket: {with_costs}")
print(f"Destinations with accommodation: {with_accom}")
print(f"Destinations with no Ryanair route: {no_route}")
print(f"\nAccommodation model: {payload['meta']['accommodation_model']['service_fee_pct']}% service, "
      f"-{payload['meta']['accommodation_model']['weekly_discount_pct']}% weekly, "
      f"validation MAE {payload['meta']['accommodation_validation']['overall_mae_pct']}%")
print(f"Wrote: {APP_JSON}  ({APP_JSON.stat().st_size/1024/1024:.2f} MB)")

Sample destination: Brussels, Belgium
  Keys: ['accommodation', 'anchor_airport', 'blurb', 'categories', 'city', 'costs', 'country', 'iata', 'id', 'iso2', 'lat', 'lon', 'no_ryanair_route', 'routes', 'tags', 'tier', 'transfer']

Destinations with fare routes: 177
Destinations with cost basket: 450
Destinations with accommodation: 450
Destinations with no Ryanair route: 11

Accommodation model: 14.0% service, -8.0% weekly, validation MAE 16.9%
Wrote: app_data\app_data.json  (0.59 MB)
